# Week 64, Day 3: Gaussian-process regression in practice

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w064/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: GP regression by Cholesky

Complete gp_fit(X, y, Xs, kernel, noise) using a Cholesky factor and triangular solves (scipy.linalg.solve_triangular), returning the predictive mean, the latent variance, and the log marginal likelihood.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cholesky, solve_triangular

k_se = lambda A, B, ell=0.8: np.exp(-0.5 * ((A[:, None] - B[None, :]) / ell) ** 2)

def gp_fit(X, y, Xs, kernel, noise):
    L = cholesky(kernel(X, X) + noise * np.eye(len(X)), lower=True)
    alpha = solve_triangular(L.T, solve_triangular(L, y, lower=True), lower=False)
    Ks = kernel(Xs, X)
    mean = Ks @ alpha
    V = ...                                            # L^-1 K_star^T
    var_f = np.diag(kernel(Xs, Xs)) - np.sum(V ** 2, axis=0)
    logml = ...
    return mean, var_f, logml

g0 = np.random.default_rng(643)
f = lambda x: np.sin(1.5 * x) + 0.3 * x
X = g0.uniform(-3, 3, 25); noise = 0.04
y = f(X) + np.sqrt(noise) * g0.normal(size=25)
Xs = np.linspace(-4, 4, 200)
mean, var_f, logml = gp_fit(X, y, Xs, k_se, noise)
sd_f, sd_y = np.sqrt(var_f), np.sqrt(var_f + noise)
plt.fill_between(Xs, mean - 2 * sd_y, mean + 2 * sd_y, alpha=0.15, label="new observation")
plt.fill_between(Xs, mean - 2 * sd_f, mean + 2 * sd_f, alpha=0.3, label="latent function")
plt.plot(Xs, mean); plt.plot(X, y, "k.", ms=8); plt.legend(); plt.show()
print(f"log marginal likelihood {logml:.3f}")

In [ ]:
import numpy as np
def test_direct():
    "the Cholesky results match direct formulas with an explicit inverse"
    Kn = k_se(X, X) + noise * np.eye(len(X)); Ki = np.linalg.inv(Kn); Ks = k_se(Xs, X)
    assert np.allclose(mean, Ks @ Ki @ y) and np.allclose(var_f, 1.0 - np.einsum("ij,jk,ik->i", Ks, Ki, Ks))
def test_logml():
    "the log marginal likelihood matches scipy's multivariate normal"
    from scipy.stats import multivariate_normal as mvn
    assert abs(logml - mvn.logpdf(y, np.zeros(len(X)), k_se(X, X) + noise * np.eye(len(X)))) < 1e-8

test_direct()
test_logml()
print("All checks passed")

### Exercise: Coverage of predictive intervals

Draw 2,000 held-out points from the same function and noise, and compute the coverage of the 95% intervals for new observations, with the noise variance set correctly (0.04) and ten times too small (0.004).

In [ ]:
gh = np.random.default_rng(6431)
Xh = gh.uniform(-3, 3, 2000)
yh = f(Xh) + np.sqrt(0.04) * gh.normal(size=2000)
cover = {}
for nv in (0.04, 0.004):
    mh, vh, _ = gp_fit(X, y, Xh, k_se, nv)
    half = 1.96 * np.sqrt(vh + nv)
    cover[nv] = ...
    print(f"assumed noise variance {nv}: coverage of 95% intervals {cover[nv]:.3f}")

In [ ]:
import numpy as np
def test_right():
    "with the true noise variance, coverage is between 0.9 and 0.99"
    assert 0.9 < cover[0.04] < 0.99
def test_wrong():
    "with the noise variance ten times too small, coverage falls below 0.8"
    assert cover[0.004] < 0.8

test_right()
test_wrong()
print("All checks passed")

### Exercise: Mapping a place field

Simulate a random-walk path that avoids the top-right corner, spike counts from a Gaussian place field, and fit a two-dimensional GP to the square-root counts (length-scale 0.12, signal variance 0.5, noise variance 0.25). Predict on a 30 by 30 grid and compare with the true field.

In [ ]:
gp_ = np.random.default_rng(6432)
steps = 400
pos = np.zeros((steps, 2)); pos[0] = [0.3, 0.3]
for t_ in range(1, steps):
    nxt = pos[t_ - 1] + 0.05 * gp_.normal(size=2)
    nxt = np.clip(nxt, 0, 1)
    if nxt[0] > 0.7 and nxt[1] > 0.7: nxt = pos[t_ - 1]                 # the corner is never entered
    pos[t_] = nxt
field = lambda P: 0.3 + 6 * np.exp(-np.sum((P - [0.4, 0.55]) ** 2, axis=-1) / (2 * 0.1 ** 2))   # spikes per bin
counts = gp_.poisson(field(pos))
ys = np.sqrt(counts) - np.sqrt(counts).mean()

k2 = lambda A, B: 0.5 * np.exp(-np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=-1) / (2 * 0.12 ** 2))
gx = np.linspace(0, 1, 30); G = np.array([[a, b] for b in gx for a in gx])
mean2, var2, _ = gp_fit(pos, ys, G, k2, 0.25)
visited = np.min(np.sum((G[:, None, :] - pos[None]) ** 2, -1), 1) < 0.05 ** 2 * 4
r_map = ...
sd2 = np.sqrt(var2)
sd_centre = sd2[np.argmin(np.sum((G - [0.4, 0.55]) ** 2, 1))]
sd_corner = sd2[np.argmin(np.sum((G - [0.95, 0.95]) ** 2, 1))]
print(f"correlation with the true field where visited: {r_map:.3f}; posterior sd at field centre {sd_centre:.2f}, in the unvisited corner {sd_corner:.2f}")
fig, ax = plt.subplots(1, 3, figsize=(10, 3))
ax[0].plot(*pos.T, lw=0.5); ax[0].set_title("path"); ax[0].set_aspect("equal")
ax[1].imshow(mean2.reshape(30, 30), origin="lower", extent=(0, 1, 0, 1)); ax[1].set_title("GP mean")
ax[2].imshow(sd2.reshape(30, 30), origin="lower", extent=(0, 1, 0, 1)); ax[2].set_title("GP sd"); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_map():
    "the GP map correlates with the true field above 0.85 within the visited region"
    assert r_map > 0.85
def test_unvisited():
    "posterior sd is larger in the unvisited corner than at the field centre"
    assert sd_corner > sd_centre

test_map()
test_unvisited()
print("All checks passed")